In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""


import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics


from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl


# define a bunch of functions 

In [ ]:
def wfall(shap_values, max_display=10, show=True):
    """ Plots an explantion of a single prediction as a waterfall plot.
    The SHAP value of a feature represents the impact of the evidence provided by that feature on the model's
    output. The waterfall plot is designed to visually display how the SHAP values (evidence) of each feature
    move the model output from our prior expectation under the background data distribution, to the final model
    prediction given the evidence of all the features. Features are sorted by the magnitude of their SHAP values
    with the smallest magnitude features grouped together at the bottom of the plot when the number of features
    in the models exceeds the max_display parameter.
    
    Parameters
    ----------
    shap_values : Explanation
        A one-dimensional Explanation object that contains the feature values and SHAP values to plot.
    max_display : str
        The maximum number of features to plot.
    show : bool
        Whether matplotlib.pyplot.show() is called before returning. Setting this to False allows the plot
        to be customized further after it has been created.
    """
    dark_o= mpl.colors.to_rgb('dimgray')
    dim_g= mpl.colors.to_rgb('darkorange')

    base_values = shap_values.base_values
    
    features = shap_values.data
    feature_names = shap_values.feature_names
    lower_bounds = getattr(shap_values, "lower_bounds", None)
    upper_bounds = getattr(shap_values, "upper_bounds", None)
    values = shap_values.values

    # make sure we only have a single output to explain
    if (type(base_values) == np.ndarray and len(base_values) > 0) or type(base_values) == list:
        raise Exception("waterfall_plot requires a scalar base_values of the model output as the first " \
                        "parameter, but you have passed an array as the first parameter! " \
                        "Try shap.waterfall_plot(explainer.base_values[0], values[0], X[0]) or " \
                        "for multi-output models try " \
                        "shap.waterfall_plot(explainer.base_values[0], values[0][0], X[0]).")

    # make sure we only have a single explanation to plot
    if len(values.shape) == 2:
        raise Exception("The waterfall_plot can currently only plot a single explanation but a matrix of explanations was passed!")
    
    # unwrap pandas series
    if safe_isinstance(features, "pandas.core.series.Series"):
        if feature_names is None:
            feature_names = list(features.index)
        features = features.values

    # fallback feature names
    if feature_names is None:
        feature_names = np.array([labels['FEATURE'] % str(i) for i in range(len(values))])
    
    # init variables we use for tracking the plot locations
    num_features = min(max_display, len(values))
    row_height = 0.5
    rng = range(num_features - 1, -1, -1)
    order = np.argsort(-np.abs(values))
    pos_lefts = []
    pos_inds = []
    pos_widths = []
    pos_low = []
    pos_high = []
    neg_lefts = []
    neg_inds = []
    neg_widths = []
    neg_low = []
    neg_high = []
    loc = base_values + values.sum()
    yticklabels = ["" for i in range(num_features + 1)]
    
    # size the plot based on how many features we are plotting
    pl.gcf().set_size_inches(8, num_features * row_height + 1.5)

    # see how many individual (vs. grouped at the end) features we are plotting
    if num_features == len(values):
        num_individual = num_features
    else:
        num_individual = num_features - 1

    # compute the locations of the individual features and plot the dashed connecting lines
    for i in range(num_individual):
        sval = values[order[i]]
        loc -= sval
        if sval >= 0:
            pos_inds.append(rng[i])
            pos_widths.append(sval)
            if lower_bounds is not None:
                pos_low.append(lower_bounds[order[i]])
                pos_high.append(upper_bounds[order[i]])
            pos_lefts.append(loc)
        else:
            neg_inds.append(rng[i])
            neg_widths.append(sval)
            if lower_bounds is not None:
                neg_low.append(lower_bounds[order[i]])
                neg_high.append(upper_bounds[order[i]])
            neg_lefts.append(loc)
        if num_individual != num_features or i + 4 < num_individual:
            pl.plot([loc, loc], [rng[i] -1 - 0.4, rng[i] + 0.4], color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
        if features is None:
            yticklabels[rng[i]] = feature_names[order[i]]
        else:
            yticklabels[rng[i]] = format_value(features[order[i]], "%0.03f") + " = " + feature_names[order[i]] 
    
    # add a last grouped feature to represent the impact of all the features we didn't show
    if num_features < len(values):
        yticklabels[0] = "%d other features" % (len(values) - num_features + 1)
        remaining_impact = base_values - loc
        if remaining_impact < 0:
            pos_inds.append(0)
            pos_widths.append(-remaining_impact)
            pos_lefts.append(loc + remaining_impact)
            c = dim_g  #colors.red_rgb
        else:
            neg_inds.append(0)
            neg_widths.append(-remaining_impact)
            neg_lefts.append(loc + remaining_impact)
            c = dark_o #colors.blue_rgb

    points = pos_lefts + list(np.array(pos_lefts) + np.array(pos_widths)) + neg_lefts + list(np.array(neg_lefts) + np.array(neg_widths))
    dataw = np.max(points) - np.min(points)
    
    # draw invisible bars just for sizing the axes
    label_padding = np.array([0.1*dataw if w < 1 else 0 for w in pos_widths])
    pl.barh(pos_inds, np.array(pos_widths) + label_padding + 0.02*dataw, left=np.array(pos_lefts) - 0.01*dataw, color=colors.red_rgb, alpha=0)
    label_padding = np.array([-0.1*dataw  if -w < 1 else 0 for w in neg_widths])
    pl.barh(neg_inds, np.array(neg_widths) + label_padding - 0.02*dataw, left=np.array(neg_lefts) + 0.01*dataw, color=colors.blue_rgb, alpha=0)
    
    # define variable we need for plotting the arrows
    head_length = 0.08
    bar_width = 0.8
    xlen = pl.xlim()[1] - pl.xlim()[0]
    fig = pl.gcf()
    ax = pl.gca()
    xticks = ax.get_xticks()
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    width, height = bbox.width, bbox.height
    bbox_to_xscale = xlen/width
    hl_scaled = bbox_to_xscale * head_length
    renderer = fig.canvas.get_renderer()
    
    # draw the positive arrows
    for i in range(len(pos_inds)):
        dist = pos_widths[i]
        arrow_obj = pl.arrow(
            pos_lefts[i], pos_inds[i], max(dist-hl_scaled, 0.000001), 0,
            head_length=min(dist, hl_scaled),
            color=dim_g, width=bar_width,
            head_width=bar_width
        )
        
        if pos_low is not None and i < len(pos_low):
            pl.errorbar(
                pos_lefts[i] + pos_widths[i], pos_inds[i], 
                xerr=np.array([[pos_widths[i] - pos_low[i]], [pos_high[i] - pos_widths[i]]]),
                ecolor=dim_g
            )

        txt_obj = pl.text(
            pos_lefts[i] + 0.5*dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                pos_lefts[i] + (5/72)*bbox_to_xscale + dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
                horizontalalignment='left', verticalalignment='center', color=dim_g,
                fontsize=12
            )
    
    # draw the negative arrows
    for i in range(len(neg_inds)):
        dist = neg_widths[i]
        
        arrow_obj = pl.arrow(
            neg_lefts[i], neg_inds[i], -max(-dist-hl_scaled, 0.000001), 0,
            head_length=min(-dist, hl_scaled),
            color=dark_o, width=bar_width,
            head_width=bar_width
        )

        if neg_low is not None and i < len(neg_low):
            pl.errorbar(
                neg_lefts[i] + neg_widths[i], neg_inds[i], 
                xerr=np.array([[neg_widths[i] - neg_low[i]], [neg_high[i] - neg_widths[i]]]),
                ecolor=dark_o
            )
        
        txt_obj = pl.text(
            neg_lefts[i] + 0.5*dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                neg_lefts[i] - (5/72)*bbox_to_xscale + dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
                horizontalalignment='right', verticalalignment='center', color=dark_o,
                fontsize=12
            )

    # draw the y-ticks twice, once in gray and then again with just the feature names in black
    ytick_pos = list(range(num_features)) + list(np.arange(num_features)+1e-8) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    pl.yticks(ytick_pos, yticklabels[:-1] + [l.split('=')[-1] for l in yticklabels[:-1]], fontsize=13)
    
    # put horizontal lines for each feature row
    for i in range(num_features):
        pl.axhline(i, color="#cccccc", lw=0.5, dashes=(1, 5), zorder=-1)
    
    # mark the prior expected value and the model prediction
    pl.axvline(base_values, 0, 1/num_features, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    fx = base_values + values.sum()
    pl.axvline(fx, 0, 1, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    
    # clean up the main axis
    pl.gca().xaxis.set_ticks_position('bottom')
    pl.gca().yaxis.set_ticks_position('none')
    pl.gca().spines['right'].set_visible(False)
    pl.gca().spines['top'].set_visible(False)
    pl.gca().spines['left'].set_visible(False)
    ax.tick_params(labelsize=13)
    #pl.xlabel("\nModel output", fontsize=12)

    # draw the E[f(X)] tick mark
    xmin,xmax = ax.get_xlim()
    ax2=ax.twiny()
    ax2.set_xlim(xmin,xmax)
    ax2.set_xticks([base_values, base_values+1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax2.set_xticklabels(["\n$E[f(X)]$","\n$ = "+format_value(base_values, "%0.03f")+"$"], fontsize=12, ha="left")
    ax2.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)

    # draw the f(x) tick mark
    ax3=ax2.twiny()
    ax3.set_xlim(xmin,xmax)
    ax3.set_xticks([base_values + values.sum(), base_values + values.sum() + 1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax3.set_xticklabels(["$f(x)$","$ = "+format_value(fx, "%0.03f")+"$"], fontsize=12, ha="left")
    tick_labels = ax3.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-10/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(12/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_color("#999999")
    ax3.spines['right'].set_visible(False)
    ax3.spines['top'].set_visible(False)
    ax3.spines['left'].set_visible(False)

    # adjust the position of the E[f(X)] = x.xx label
    tick_labels = ax2.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-20/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(22/72., -1/72., fig.dpi_scale_trans))
    
    tick_labels[1].set_color("#999999")

    # color the y tick labels that have the feature values as gray
    # (these fall behind the black ones with just the feature name)
    tick_labels = ax.yaxis.get_majorticklabels()
    for i in range(num_features):
        tick_labels[i].set_color("#999999")
    
    if show:
        pl.show()

def dbscan_plot(data,eps=0.1,min_samples=50):
    X=data
    X = StandardScaler().fit_transform(X)
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
    core_samples_mask[db.core_sample_indices_] = True
    labels = db.labels_

    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    print('Estimated number of clusters: %d' % n_clusters_)
    print('Estimated number of noise points: %d' % n_noise_)
    print("Silhouette Coefficient: %0.3f"
          % metrics.silhouette_score(X, labels))

    # Black removed and is used for noise instead.
    plt.figure(figsize=(10, 10))
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each)
              for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]

        class_member_mask = (labels == k)
        
        xy = X[class_member_mask & core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),label = k,
                 markeredgecolor='k', markersize=14)
        
        xy = X[class_member_mask & ~core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                 markeredgecolor='k', markersize=6)
    
    plt.legend(fontsize=15, title_fontsize='40')    
    plt.title('Estimated number of clusters: %d' % n_clusters_)
#    plt.show()
    return labels



def residual(params, x, data):
    alpha = params['alpha']
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H3.3']*alpha+x['H4']*beta+x['H3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H3.3'])+np.std(od['H4'])+np.std(od['H3'])


def residual2(params, x, data):
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H4']*beta+x['H3.3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H4'])+np.std(od['H3.3'])



def twoSampZ(X1, X2):
    from numpy import sqrt, abs, round
    from scipy.stats import norm
    mudiff=np.mean(X1)-np.mean(X2)
    sd1=np.std(X1)
    sd2=np.std(X2)
    n1=len(X1)
    n2=len(X2)
    pooledSE = sqrt(sd1**2/n1 + sd2**2/n2)
    z = ((X1 - X2) - mudiff)/pooledSE
    pval = 2*(1 - norm.cdf(abs(z)))
    return round(pval, 4)

def statistic(dframe):
    return dframe.corr().loc[Var1,Var2]


def draw_umap(data,n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean', title=''
              ,cc=0,rstate=42,dens=False):
    fit = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric, random_state=rstate, verbose=True, densmap=dens
    )
    u = fit.fit_transform(data);
    plt.figure(figsize=(6, 5))
    if n_components == 2:
        plt.scatter(u[:,0], u[:,1], c=cc,s=3,cmap=plt.cm.seismic)
        plt.clim(-5,5)
        plt.colorbar()
    plt.title(title, fontsize=18)
    return u;


def NormMark(data):
    params = Parameters()
    params.add('beta', value=0.1, min=0)
    params.add('gamma', value=0.1, min=0)
    params.add('alpha', value=0.1, min=0)
    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value
    alpha=out.params['alpha'].value
    avMarkers=ddf['H3.3']*alpha+ddf['H4']*beta+ddf['H3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data

def NormMark2(data):
    params = Parameters()
    params.add('beta', value=0.1, min=-1000)
    params.add('gamma', value=0.1, min=-1000)

    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual2, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value

    avMarkers=ddf['H4']*beta+ddf['H3.3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols_M]=data[EpiCols_M]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data






def f(): raise Exception("Found exit()")



def BPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.boxplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   

def VPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.violinplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   


def KPlots(data,NMS,titleSup=''):
    for NN in NMS:
        plt.figure(figsize=(10,10))
        sns.kdeplot(data=data,x=NN,color='blue')
        
#        plt.legend()
        plt.title(""+NN+" "+titleSup)
        plt.show()



def MeanDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

    
def MedDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].median().sort_values(ascending=False)
    dd1=data2[Markers].median().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)    
    
def MeanDistIdU(data1,data2,Markers,title=''):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    
    colors = ['dodgerblue' if x < 0 else 'darkmagenta' for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

def KPlot_Mrk(Mark,titleSup=''):
    plt.figure(figsize=(10,10))
    sns.kdeplot(data=C01,x=Mark,label="C01")
    sns.kdeplot(data=C02,x=Mark,label="C02")
    sns.kdeplot(data=C03,x=Mark,label="C03")
    sns.kdeplot(data=C04,x=Mark,label="C04")
    sns.kdeplot(data=C05,x=Mark,label="C05")
    plt.legend()
    plt.title(""+Mark+" "+titleSup)
    plt.show()
    
    
    
    

def UMAP_Plot(data1,data2,Markers,Set1='C01',Set2='Other',titleSup=''):
    data1=data1.assign(Set=Set1)
    data2=data2.assign(Set=Set2)
    CAll=data1.append(data2).sample(frac=0.1).copy()
    print(CAll)
    X_2d=draw_umap(CAll[Markers],cc=CAll['H3'],min_dist=0.01)
    for NN in NamesAll:
        cc=CAll[NN]#[mask]
        plt.figure(figsize=(6, 5))
        plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                    c=cc, cmap=plt.cm.jet)
    #    cmap = matplotlib.cm.get_cmap('jet')
        plt.colorbar()
    #    plt.clim(-3.5,3.5)
        plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    #    mask=CAllmask[TSNEVar]==True
    #    rgba = cmap(-10)
    #    plt.scatter(X_2d[mask][:,0],X_2d[mask][:,1],s=2,
    #                color=rgba) 
        plt.title(NN+" "+titleSup)
        plt.show()

    plt.figure(figsize=(6, 5))
    mask=CAll.Set==Set1
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='blue', label=Set1)        
    mask=CAll.Set==Set2
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='red', label=Set2)        
    plt.legend()
    plt.show()
       

def DeltaCorr(data1,data2,Markers,titleSup=''):
    params = {'axes.titlesize': 30,
              'legend.fontsize': 20,
              'figure.figsize': (16, 10),
              'axes.labelsize': 20,
              'axes.titlesize': 20,
              'xtick.labelsize': 16,
              'ytick.labelsize': 16,
              'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

    print(titleSup)
    plt.figure(figsize=(20,20))
    matrix=data2[Markers].corr()-data1[Markers].corr()
    g=sns.clustermap(matrix, annot=True, annot_kws={"size":8},
                     cmap=plt.cm.jet,vmin=matrix.min().min(),vmax=matrix.max().max(),linewidths=.1); 
    plt.xticks(rotation=0); 
    plt.yticks(rotation=0); 

    plt.title(titleSup)
    plt.show()
    
    
def DefStyle():
    params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

# Load and initialize

In [ ]:
NamesAll=['CD45',
 'H3',
 'K5',
 'EpCam',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3.3',
 'H3K64ac',
 'BMI-1',
 'ZEB1',
 'H4',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'H3K36me3',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3S28p',
 'Ir_DNA2',
 'Live_Dead']


EpiCols=[
 'H3',
 'H3K27me2',
 'H3K4me3',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'H3.3',
 'H3K64ac',
 'H4',
 'H3K27ac',
 'H4K20me3',
 'H3K36me3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'H3S28p',
]

CellIden=[
 'CD45',
 'K5',
 'EpCam',
 'aSMA',
 'Vimentin',
 'ZEB1',
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'CD44',
 'K8-18',
]


ToNorm=[
 'H3',
 'K5',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3.3',
 'H3K64ac',
 'BMI-1',
 'ZEB1',
 'H4',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'H3K36me3',
 'GATA3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'Ki67',
 'K8-18',
 'H3S28p',
 ]

NMS=['CD45',
 'K5',
 'EpCam',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3K64ac',
 'BMI-1',
 'ZEB1',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'H3K36me3',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3S28p',
]

In [ ]:
dir="~/Dropbox/CyTOF_Breast/Kaplan_1st/"
K1=pd.read_csv(dir+"BCK-01_noaf_18Sep2022_01_0.fcs_file_internal_comp_residual.csv")
K2=pd.read_csv(dir+"BCK-02_noaf_18Sep2022_01_0.fcs_file_internal_comp_residual.csv")


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

K1=K1[NamesAll]
K2=K2[NamesAll]



In [ ]:
sns.kdeplot(K1['H4'],c='r')
sns.kdeplot(K1['H3'],c='g')
sns.kdeplot(K1['H3.3'],c='b')
plt.xlim([-10,500])

# Gate on H3.3/H4 too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4']#,'H3']#,'H3']
# #Ly7
# print("C01 Ly7")
# print(len(C01),len(C01))
# C01=C01[(C01[GateColumns]>5).all(axis=1)]
# print(len(C01),len(C01))
# C01=C01[(C01<np.quantile(C01,0.9999,axis=0)).all(axis=1)]
# print(len(C01),len(C01))

# #EZH2
# print("C03 EZH2")
# print(len(C03))
# C03=C03[(C03[GateColumns]>5).all(axis=1)]
# print(len(C03))
# C03=C03[(C03<np.quantile(C03,0.9999,axis=0)).all(axis=1)]
# print(len(C03))


def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data


K1=Gate(K1,"K1")
K2=Gate(K2,"K2")

In [ ]:

scFac=5
K1=np.arcsinh(K1/scFac)
K2=np.arcsinh(K2/scFac)


for M in NamesAll:
    plt.figure()
    sns.kdeplot(K1[M],c='r',label='Tumor 1')
    sns.kdeplot(K2[M],c='g',label='Tumor 2')
   
    plt.legend()
    plt.title('ArcSinh Unnormalized')
    plt.savefig('Plots/Hist_'+M+'.png')
    plt.show()

In [ ]:
ThK1={'CD45': 1.8686868686868687}
ThK2={'CD45': 2.1717171717171717}


In [ ]:
K1CD45Neg=K1[K1.CD45<ThK1['CD45']].copy()
K2CD45Neg=K2[K2.CD45<ThK2['CD45']].copy()


# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return (d.std()['H3.3']+d.std()['H4']+d.std()['H3'])**2

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.3,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf, ddf,Q,M,M1,M2),method='cg')
    AA=out.params['a'].value

    M=M1*AA+M2*(1-AA)
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[EpiCols]=data[EpiCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:
EpiBCK=EpiCols.copy()
EpiCols=ToNorm.copy()






print("K1")
print(K1.std()['H3.3']+K1.std()['H4']+K1.std()['H3'])
K1=NormalizeNew2(K1)
print(K1.std()['H3.3']+K1.std()['H4']+K1.std()['H3'])

print("K2")
print(K2.std()['H3.3']+K2.std()['H4']+K2.std()['H3'])
K2=NormalizeNew2(K2)
print(K2.std()['H3.3']+K2.std()['H4']+K2.std()['H3'])


K1CD45Neg=NormalizeNew2(K1CD45Neg)
K2CD45Neg=NormalizeNew2(K2CD45Neg)

EpiCols=EpiBCK.copy()


In [ ]:
from tqdm import tqdm
Mean_Core=K1[['H3.3','H4']].mean(axis=1)
for N in tqdm(ToNorm):
    K1[N]=K1[N]/Mean_Core
    
    
from tqdm import tqdm
Mean_Core=K2[['H3.3','H4']].mean(axis=1)
for N in tqdm(ToNorm):
    K2[N]=K2[N]/Mean_Core

from tqdm import tqdm
Mean_Core=K1CD45Neg[['H3.3','H4']].mean(axis=1)
for N in tqdm(ToNorm):
    K1CD45Neg[N]=K1CD45Neg[N]/Mean_Core
    
    
from tqdm import tqdm
Mean_Core=K2CD45Neg[['H3.3','H4']].mean(axis=1)
for N in tqdm(ToNorm):
    K2CD45Neg[N]=K2CD45Neg[N]/Mean_Core

In [ ]:
aaaa=pd.concat([K1]).copy()
m=np.mean(aaaa)
s=np.std(aaaa)
K1=(K1-m)/s

aaaa=pd.concat([K2]).copy()
m=np.mean(aaaa)
s=np.std(aaaa)
K2=(K2-m)/s

aaaa=pd.concat([K1CD45Neg]).copy()
m=np.mean(aaaa)
s=np.std(aaaa)
K1CD45Neg=(K1CD45Neg-m)/s

aaaa=pd.concat([K2CD45Neg]).copy()
m=np.mean(aaaa)
s=np.std(aaaa)
K2CD45Neg=(K2CD45Neg-m)/s


print(aaaa.std())

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

In [ ]:
Lines=['Tumor1','Tumor2']

In [ ]:
print(len(K1),len(K2))
print(len(K1CD45Neg),len(K2CD45Neg))

# UMAP Tumor 2 - ALL

## Cell Identity

In [ ]:
np.random.seed(101)
idx=np.random.choice(K2.index,size=20000)


In [ ]:
CAll=pd.concat([K2.loc[idx]]).copy()

In [ ]:
#CellIden.remove('CD45')
X_2d=draw_umap(CAll[CellIden],cc=CAll['CD45'],min_dist=0.05,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 Cell Iden UMAP - "+TSNEVar)
    plt.savefig('Plots/Tumor2_UMAP_CellIdentity_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.1,min_samples=110)

In [ ]:
m=labels!=-1

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2=CAll.copy()

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels

In [ ]:
sc.pp.neighbors(K2AN)
sc.tl.umap(K2AN,n_components=3)
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=12, legend_fontoutline=2,frameon=True,
               title='Clusters Tumor 2 - Cell Iden Based', palette=['r','orange','yellow','b'],show=False,projection='2d',)
plt.savefig("Plots/Clust_T2_CellIden.png")

In [ ]:
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NamesAll]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,20), annot_kws={"size":8}, center=0,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('T2 Cell Iden Based') 
plt.savefig('Plots/T2_CellIden.png')

## Epigenetics Based

In [ ]:
MRK=EpiCols.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')
X_2d=draw_umap(CAll[MRK],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 Epigen UMAP - "+TSNEVar)
    plt.savefig('Plots/Tumor2_UMAP_Epi'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.2,min_samples=120)

In [ ]:
m=labels!=-1

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels

In [ ]:
sc.pp.neighbors(K2AN)
sc.tl.umap(K2AN,n_components=2)
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=16, legend_fontoutline=4,frameon=True,
               title='Clusters Tumor 2 - Epigen Based', palette=['r','orange','yellow','b'],show=False,projection='2d',)
plt.savefig("Plots/Clust_T2_Epigen.png")

In [ ]:
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NamesAll]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,20), annot_kws={"size":8}, center=0,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('T2 Epigen Based') 
plt.savefig('Plots/T2_Epigen.png')

## All Markers

In [ ]:
MRK=[
 'CD45',
 'K5',
 'EpCam',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3K64ac',
 'BMI-1',
 'ZEB1',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'H3K36me3',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3S28p',
]

In [ ]:

X_2d=draw_umap(CAll[MRK],cc=CAll['H4'],min_dist=0.05,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 All Markers UMAP - "+TSNEVar)
    plt.savefig('Plots/Tumor2_UMAP_All'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.1,min_samples=100)

In [ ]:
m=labels!=-1

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels

In [ ]:
sc.pp.neighbors(K2AN)
sc.tl.umap(K2AN,n_components=2)
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=16, legend_fontoutline=4,frameon=True,
               title='Clusters Tumor 2 - All MRK Based', palette=['r','orange','yellow','b','indigo','magenta'],show=False,projection='2d',)
plt.savefig("Plots/Clust_T2_All.png")

In [ ]:
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NamesAll]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,20), annot_kws={"size":8}, center=0,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('T2 All MRK Based') 
plt.savefig('Plots/T2_All.png')

# UMAP Tumor 2 - CD 45 Negative

In [ ]:
K2Bck=K2.copy()
K2=K2CD45Neg.copy()

## Cell Identity

In [ ]:
np.random.seed(42)
idx=np.random.choice(K2.index,size=20000)
CAll=pd.concat([K2]).copy()

In [ ]:
try:
    CellIden.remove('CD45')
except:
    pass
X_2d=draw_umap(CAll[CellIden],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
X_2d.shape

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 Cell Iden UMAP - "+TSNEVar)
    plt.savefig('Plots/CD45Neg_Tumor2_UMAP_CellIdentity_'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.041,min_samples=30)
#labels=dbscan_plot(X_2d,eps=0.045,min_samples=30)

In [ ]:
labels[labels==5]=0
labels[labels==4]=3

In [ ]:
K2=CAll.copy()

In [ ]:
m=labels!=-1
#m=labels==2

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels
sc.pp.neighbors(K2AN)

In [ ]:
#sc.tl.leiden(K2AN,resolution=0.2)

In [ ]:
# with rc_context({'figure.figsize': (10, 10)}):
#     sc.pl.umap(K2AN, color=['leiden','clust'], add_outline=True, legend_loc='on data',
#                legend_fontsize=16, legend_fontoutline=4,frameon=True,cmap=plt.cm.seismic,
#                title='Clusters Tumor 2 - Leiden', show=False,projection='2d',)

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
#K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=16, legend_fontoutline=4,frameon=True,
               title='Clusters Tumor 2 - Cell Iden Based', palette=['r','orange','yellow','b'],show=False,projection='2d',)
#plt.savefig("Plots/Clust_T2_CellIden.png")

In [ ]:
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(K2AN, color=NamesAll+['clust'],ncols=5,vmax='p95',vmin='p5',
           cmap=plt.cm.seismic,add_outline=True,show=False)
    plt.savefig("Plots/Tumor2.png",dpi=200,bbox_inches='tight')


In [ ]:
CellIden.append('CD45')
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NMS]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,20), annot_kws={"size":14}, center=0,col_cluster=False,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('CD45- T2 Cell Iden Based') 
#plt.savefig('Plots/T2_CellIden.png')

In [ ]:
EP=EpiCols.copy()
EP.remove('H3')
EP.remove('H3.3')
EP.remove('H4')
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[EP]
amin=Mat[EP].min().min()
amax=Mat[EP].max().max()
g=sns.clustermap(Mat[EP].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,15), annot_kws={"size":12}, center=0,yticklabels=True,col_cluster=False,row_cluster=True,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('T2 Cell Iden Based (45-)') 
#plt.savefig('Plots/T2_CellIden_CD45Neg_Epi.png')

In [ ]:
MeanDist(K2[K2.Clust==0],K2[K2.Clust==1],EP)

In [ ]:
m=K2.Clust.isin([0,1])

In [ ]:
K2Sub=K2[m].copy()

In [ ]:
MRK=EpiCols.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')
X_2d_Sub=draw_umap(K2Sub[MRK],cc=K2Sub['CD24'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
m=K2Sub.Clust==0
plt.scatter(X_2d_Sub[m,0],X_2d_Sub[m,1],s=1,c='red')
m=K2Sub.Clust==1
plt.scatter(X_2d_Sub[m,0],X_2d_Sub[m,1],s=1,c='blue')

In [ ]:
K2AN.X

In [ ]:
K=K2AN[:,CellIden].copy()

K.obs['clust']=(K2[m].Clust.astype('category').values)

In [ ]:
K.obs['cl']=K.obs['clust'].astype('string')

In [ ]:
sc.tl.rank_genes_groups(K, groupby='cl', method='wilcoxon',n_genes=K2AN.shape[1])


In [ ]:
K.obs['cl']

In [ ]:
sc.tl.paga(K2AN, groups='clust')

In [ ]:
sc.pl.paga(K2AN, color=['clust'])

In [ ]:
ax = sc.pl.rank_genes_groups_violin(K,gene_names=CellIden,strip=False)

## Epigenetics Based

In [ ]:
MRK=EpiCols.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')
X_2d=draw_umap(CAll[MRK],cc=CAll['H4'],min_dist=0.01,n_neighbors=150,rstate=42)
plt.show()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 Epigen UMAP - "+TSNEVar)
    plt.savefig('Plots/Tumor2_UMAP_Epi'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.11,min_samples=40)

In [ ]:
m=labels!=-1

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels

In [ ]:
sc.pp.neighbors(K2AN)
sc.tl.umap(K2AN,n_components=2)
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=16, legend_fontoutline=4,frameon=True,
               title='Clusters Tumor 2 - Epigen Based', palette=['r','orange','yellow','b'],show=False,projection='2d',)
plt.savefig("Plots/Clust_T2_Epigen.png")

In [ ]:
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NMS]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                 annot_kws={"size":8}, center=0,
                annot=True, linewidths=1,linecolor='k',figsize=(10,20))
g.ax_col_dendrogram.set_title('T2 Epigen Based') 
plt.savefig('Plots/T2_Epigen.png')

## All Markers

In [ ]:
MRK=[
 'CD45',
 'K5',
 'EpCam',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3K64ac',
 'BMI-1',
 'ZEB1',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'H3K36me3',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3S28p',
]

In [ ]:

X_2d=draw_umap(CAll[MRK],cc=CAll['CD45'],min_dist=0.01,n_neighbors=100,rstate=42)
plt.show()

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title("Tumor 2 All Markers UMAP - "+TSNEVar)
    plt.savefig('Plots/Tumor2_UMAP_All'+NN+'.png',dpi=200,bbox_inches='tight')

    plt.show()

In [ ]:
labels=dbscan_plot(X_2d,eps=0.07,min_samples=50)

In [ ]:
m=labels!=-1

In [ ]:
import scanpy as sc
import anndata

In [ ]:
K2AN=anndata.AnnData(K2[m],dtype=np.float32)

In [ ]:
K2['Clust']=labels

In [ ]:
sc.pp.neighbors(K2AN)
sc.tl.umap(K2AN,n_components=2)
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obsm['X_umap']=X_2d[m]

In [ ]:
K2AN.obs['clust']=K2[m].Clust.astype('category').values

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(K2AN, color='clust', add_outline=True, legend_loc='on data',
               legend_fontsize=16, legend_fontoutline=4,frameon=True,
               title='Clusters Tumor 2 - All MRK Based', palette=['r','orange','yellow','b','g','indigo','magenta'],show=False,projection='2d',)
plt.savefig("Plots/Clust_T2_All.png")

In [ ]:
Mat=K2[K2.Clust!=-1].groupby(by='Clust').mean()[NMS]
amin=Mat[NMS].min().min()
amax=Mat[NMS].max().max()
g=sns.clustermap(Mat[NMS].T,cmap=plt.cm.seismic,vmin=amin,vmax=amax,
                figsize=(10,20), annot_kws={"size":8}, center=0,
                annot=True, linewidths=1,linecolor='k',)
g.ax_col_dendrogram.set_title('T2 All MRK Based') 
plt.savefig('Plots/T2_All.png')